In [ ]:
from astropy.table import Table
table_current = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/full_table.csv')
import numpy as np
import glob
files = np.concatenate([glob.glob('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/ngc1433*nircam*.csv'),
glob.glob('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/ngc1512*nircam*.csv'),
glob.glob('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/ngc1672*nircam*.csv')])
files = np.append(files, '/project/galaxies/tjuchau/data_files/misc_data/clusters.csv')
cols = Table.read(files[0]).colnames
t = Table.read(files[0])
for file in files:
    table = Table.read(file)
    if cols == table.colnames:
        print('all cols match')
    print(len(table.colnames))
test_table = Table.read('/project/galaxies/tjuchau/data_files/Kiana_Cluster_Files/test_full_table.csv')
print(len(test_table))
print(len(table_current))

In [ ]:
import os
import subprocess


slug_output_dir = '/project/galaxies/tjuchau/data_files/misc_data/slug_outputs'

param_file = "/project/galaxies/tjuchau/projects/EW_vs_Age/modeling_files/cluster_library.param"

with open(param_file, "w") as f:
    f.write("""
# Basic setup
model_name cluster_lib
out_dir {slug_output}
sim_type cluster
n_trials 50000

# Cluster properties
cluster_mass cmf
imf kroupa.imf

# Time sampling
log_time 1
time_step 0.1
start_time 1e5
end_time 1e8

# Metallicity
metallicity 0.143

# Nebular emission (IMPORTANT for F187N)
compute_nebular 1
nebular_phi 0.7

# Extinction
A_V 0.0

# Photometry filters (you MUST define JWST filters)
phot_mode L_nu
phot_bands JWST_F150W, JWST_F187N, JWST_F200W, JWST_F300M

# Output
out_cluster_phot 1
out_cluster 1
output_mode binary
""")
slug_path = "/project/galaxies/tjuchau/software/Slug/slug2/bin/slug"

# Run SLUG
subprocess.run([f"{slug_path}", param_file])

In [ ]:
import numpy as np
from slugpy import read_cluster
from slugpy.bayesphot import bp

# Load SLUG library
lib = read_cluster("slug_output/cluster_lib_cluster")

# Extract photometry
phot = lib['phot']  # shape: (N_models, N_filters)

# Define physical parameters to infer
# (these are stored in SLUG output)
phys = np.vstack([
    lib['mass'],
    lib['age'],
    lib['A_V']
]).T

# Build Bayesian model
model = bp(phys, phot)